In [1]:
import numpy as np
from numpy import random
import itertools

In [107]:
class NK_model(object):
    
    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.start_type = 0    # the initial given start genotype
        self.get_epistic_site()  # get the epistic network among the sites (including the site itself) 
        
        self.allelic_each_site = [[] for _ in range(self.n)]   # will be used to store the episitc combinations at each site
        self.fitness_list = [{} for _ in range(self.n)]   # will be used to store the fitness of epistic combinations at each
                                                          # site
        self.genotype_list = []   # will be used to store the genotypes generated (subset of all the genotypes)
        self.geno_fitness = []   # will be used to store the calculated fitness of genotypes generated
        self.landscape = {}  # will be used to store the generated landscape (subset of the whole landscape)
        
        
        
    @staticmethod
    def generate_start_type(seq_length):
        '''generate a genotype as the start_type
        
        parameters: 
        ----------
            seq_length: int, the length of the wanted genotype sequence
        output: 
        -------
            str that represents the start_type
        '''
        
        a_list = []
        for i in range(seq_length):  # generate a list randomly composed of 0 and 1
            a_list.append(random.randint(0, 2))
            
        start_type_list = []   
        for i in a_list:   # convert the integer elements (0 and 1) in a_list into string elements ('0' and '1') and append 
            start_type_list.append(str(i))   # into start_type_list
            
        start_type = ''.join(start_type_list)  # join the elements in start_type_list into a string
        
        return start_type
    
    
    @staticmethod
    def creat_neighbor(genotype, mutant_num):
        '''creat the mutant_num-mutant neighbors of the given genotype
        
        parameters: 
        ----------
            genotype: str that repesents genotype, eg: "01", "00"
            mutant_num: int, the number of mutant site to creat neighbors
            
        output: 
        -------
            list that store the mutant_num-mutant neighbors of the given genotype
        '''
        
        total_neigh_list= []  # will be used to store the created neighbors of the given genotype
    
        geno_list=list(genotype) # convert the given genotype (str) into a list
    
        indices = list(range(len(geno_list))) # get all the indices for the given genotype
    
        mutant_indices = list(itertools.combinations(indices, mutant_num)) # get all possible combinations of picking 
                                                                    # mutant_num intergers from indices (i.e. mutant sites)
    
        for i in mutant_indices:  # get all its mutant_num -mutant neighbors of the given genotype
            neigh_list = geno_list.copy()
            for j in i:
                if geno_list[j] == '0':
                    neigh_list[j] = '1'
                else:
                    neigh_list[j] = '0'
            
            total_neigh_list.append(''.join(neigh_list))
    
        return total_neigh_list
    
    
    def get_epistic_site(self):
        '''Generate the epistic networks among the sites'''

        self.epistic_site = []
        for i in range(self.n):
            all_site = [x for x in range(self.n) if x != i]  # get the indexes of all sites excluding i
            
            indices = set()   # will be used to store the indexes of epistic sites
            while len(indices) < self.k:
                indices.add(random.randint(0, len(all_site))) # randomly pick the indexes of k elements from all_site

            indices = list(indices)  # convert the set into list
            
            one_site = [i]   # first append the site under study into one_site (a little bit different from 0615 code)
            for j in indices:  # get the epistic sites of site i
                one_site.append(all_site[j])
            
            self.epistic_site.append(one_site) 
           
        
    @staticmethod     
    def get_epistic_combination(genotype, network):
        '''generate the epistic pairs in a given genotype according to the network
        
        parameters: 
        ----------
            genotype: str that repesents a genotype, 
            network: nested list that stores the epistic networks among the sites (self.epistic_site)
            
        output: 
        -------
            list that stores all the epistic combinations in the given genotype
        '''
        
        epistic_combination = []   # will be used to store the epistic combinations in the given genotype
        
        for j in range(len(genotype)):
            one_site = []   
            for i in network[j]:   # get all epistic alleles in one site (store in list one_site)
                one_site.append(genotype[i])
                
            epistic_combination.append(''.join(one_site))  # combine the epistic alleles and store into epistic_combination
                
        return epistic_combination
        
        
    @staticmethod      
    def cal_fitness(epistic_pair, fitness_list):
        '''calculate the fitness of a given genotype according to the fitness_list (here the epistic_pair is generate by
        the genotype, i.e. the epistic_combination in get_epistic_combination(genotype, network) method)
        
        parameters: 
        ----------
            epistic_pair: list that stores the epistic combinations in a genotype  
            fitness_list: list that stores the dictionary which indicates the fitness of each epistic combinations 
                          in each site
            
        output: 
        -------
            float, the calculated fitness of a genotype (the genotype which corresponded to epistic_pair)
        '''
        
        one_fitness = []    
        
        for i in range(len(epistic_pair)):  # fitness_list[i] is the fitness dictionary that corresponds to site i.  
            one_fitness.append(fitness_list[i][epistic_pair[i]])  # epistic_pair[i] is the epistic combination in site i (the
                                                                # key in the dictionary)
        i_fitness = sum(one_fitness)/len(one_fitness)  # calculate the fitness
            
        return i_fitness
            

        
    def random_walk(self):
        '''Simulate one step  of random walk along the landscape'''
        
        if self.start_type==0:  # self.start_type==0 (the initial given value), means initial step, randomly generates a 
                                # genotype as start_type (length is self.n)
            self.start_type = self.generate_start_type(self.n)
        else:  # start!='', not initial step, just pass
            pass
        
        neighbor_list = [self.start_type] # this list will be used to store the self.start_type and all its neighbors
                                        # (next step will select from the neighbors of self.start_type)
        
        neighbor_list += self.creat_neighbor(self.start_type, 1) # get all 1-mutant neighbors of start_type
                
        all_epistic_pair = [] # this list will be used to store all epistic combinations for genotypes in neighbor_list
        
        for i in neighbor_list:  # i is genotype in neighbor_list, self.epistic_site is network among the site
            all_epistic_pair.append(self.get_epistic_combination(i, self.epistic_site))
        
        
        allelic_set = []  # list that will be used to store all the epistic combinations in site 0, 1, 2, ..., self.n-1 
                        # (use set to exclude the repeat ones) for genotypes in neighbor_list
        
        for j in range(self.n): 
            pos_allelic_pair = [] # will be used to store all epistic combinations in site j for genotypes in neighbor_list
            for s in all_epistic_pair:
                pos_allelic_pair.append(s[j])   
            
            pos_allelic_set = list(set(pos_allelic_pair))  # convert the pos_allelic_pair to set then convert back to list to
                                                    # get the non-repeated epistic combinations in site j
                
            allelic_set.append(pos_allelic_set)  # append the non-repeated epistic combinations in site j into allelic_set to
                                        # get epistic combinations in all sites
            
        for s in range(self.n): # the length of allelic_set is self.n
            for i in allelic_set[s]: # i is the allelic combinations in site s, self.allelic_each_site is used to store the 
                                    #  episitc combinations at each site, self.allelic_each_site[s] is episitc combinations 
                                    # at site s that has already been generated
                if i not in self.allelic_each_site[s]: # if i is not in self.allelic_each_site[s], means epistic combination i
                                 # has not been generated, just append i into self.allelic_each_site[s] and generate a fitness
                                 # for i and store into self.fitness_list[s](i will be the key)
                    self.allelic_each_site[s].append(i)
                    self.fitness_list[s][i] = random.random(1)[0]
            
        w_fitness = []   # will be used to store the calculated fitness of each genotype in neighbor_list
        
        for i in neighbor_list:
            if i not in self.genotype_list: # i not in self.genotype_list, means the fitness of genotype i has not been
                                            # calculated--> need to be calculated this time
                index_i = neighbor_list.index(i) # first get the index of i (this index is the same as the epistic 
                                                # combinations for genotype i in all_epistic_pair)
                i_fitness = self.cal_fitness(all_epistic_pair[index_i], self.fitness_list)
                w_fitness.append(i_fitness)
            else:  # otherwise it means that the fitness of genotype i has been calculated, just find it from the landscape
                w_fitness.append(self.landscape[i])
            
        neigh_fitness = dict(zip(neighbor_list[1:], w_fitness[1:]))  # creat a dictionary to store the genotypes and fitness of 
                                                            # the 1-mutant neighbors of self.start_type (start_type is not 
                                                            # included as it won't be selected for self.next_step) 
                                                            # key: genotype; value: fitness
    
        self.start_type_fitness = w_fitness[0]  # get the fitness of self.start_type
        
        possible_neigh = {k : v for k, v in neigh_fitness.items() if v >= self.start_type_fitness}  # get all possible   
                                                                    # neighbors for self.next_step (only if the fitness >= 
                                                                    # self.start_type fitness can be possible neighbors 
                                                                    # for self.next_step) 
        
        if possible_neigh:   # if possible_neigh is not empty, it means that possible genotypes for self.next_step exist;
                             # select the genotype for self.next_step according to their fitness
            pop_gens = list(possible_neigh.keys())   # get all the possible genotypes of self.next_step
            self.next_type = pop_gens[random.randint(0, len(pop_gens))] # randomly select one genotype as self.next_step
            self.next_type_fitness = possible_neigh[self.next_type] # get the fitness of self.next_step
                                                                                
        else: # if possible_neigh is empty, it means that there if no option for self.next_step-->cannot evolve
              #-->reach local optima
            self.next_type = ''
            self.next_type_fitness = 0   
           
    
        self.genotype_list.extend(neighbor_list)  # updating the self.genotype_list by storing all the genotype in neighbor_list
                                                # into self.genotype_list
            
        self.genotype_list = list(set(self.genotype_list))  # exclude the repeating genotypes in self.genotype_list
        
        landscape_subset = dict(zip(neighbor_list, w_fitness)) # the subset of landscape generated in this random_walk step
        
        self.landscape.update(landscape_subset)  # update self.landscape by the subset of landscape generated in this random
                                                 # _walk step
                
        
    def repeat_random_walk(self):
        '''Simulate the random walk in the landscape'''
        
        self.walk_list = []  # this list will be used to store the steps of random walk
        
        fit = self.random_walk()  # run the random_walk method
        
        self.walk_list.append([self.start_type, self.start_type_fitness])  # the initial step
        
        while self.next_type and self.next_type_fitness:  # this means that while next_step!='' and fitness[next_step]!=0.0,  
                                                         #not reach local optima
            self.walk_list.append([self.next_type, self.next_type_fitness]) # the next_step
            self.start_type = self.next_type  # update the self.start_type and self.start_type_fitness
            self.start_type_fitness = self.next_type_fitness
            fit = self.random_walk() # run the random_walk method with the updated self.start_type 
        
        
    def multiple_random_walk(self, times):
        '''Allow multiple random walks in a landscape'''
        
        self.multiple_walk_list = []  # will be used to store the walk
        
        for i in range(times):
            self.repeat_random_walk()   # 
            self.multiple_walk_list.append(self.walk_list)
            self.start_type = self. generate_start_type(self.n)  # generate the start_type again

In [112]:
a = NK_model(4,1)
print(a.epistic_site)
# a.random_walk()
# a.repeat_random_walk()
a.multiple_random_walk(3)
print(a.multiple_walk_list)
print(a.landscape)

[[0, 2], [1, 2], [2, 0], [3, 0]]
1 0101
2 ['0101', '1101', '0001', '0111', '0100']
3 [['00', '10', '00', '10'], ['10', '10', '01', '11'], ['00', '00', '00', '10'], ['01', '11', '10', '10'], ['00', '10', '00', '00']]
7 [{'00': 0.61721189265771792, '01': 0.43342844015341742, '10': 0.077739083557002875}, {'00': 0.13592040713610065, '10': 0.76155803662244148, '11': 0.1516818829411346}, {'00': 0.32657580930506713, '01': 0.1970875695009684, '10': 0.50145942698018997}, {'00': 0.21399451112498991, '10': 0.71584324512393815, '11': 0.6046846696784991}]
9 [0.60529724592729117, 0.41026733983972796, 0.44888783855570591, 0.45060324879967006, 0.47983506242755414]
10 {'0111': 0.45060324879967006, '0100': 0.47983506242755414, '0001': 0.44888783855570591, '1101': 0.41026733983972796}
11 {}
12 
13 {'0100': 0.47983506242755414, '0111': 0.45060324879967006, '0001': 0.44888783855570591, '0101': 0.60529724592729117, '1101': 0.41026733983972796}
14 [['0101', 0.60529724592729117]]
1 0100
2 ['0100', '1100', '00

array([], dtype=float64)